In [ ]:
import os
print(os.listdir())

In [ ]:
import pandas as pd
df=pd.read_csv("PhonePe_Statement_Oct2025_Oct2026.csv",skiprows=3,dtype=str)
df.head()

In [ ]:
df=df[df["Transaction ID"].str.startswith("T", na=False)]
df["Amount"]=pd.to_numeric(df["Amount"])
df["Date"]= pd.to_datetime(df["Date"],format="%b %d, %Y")
print(len(df))
df.dtypes

In [ ]:
df["Month"] = df["Date"].dt.to_period("M")
spending = df[df["Transaction Type"] == "DEBIT"]
monthly = spending.groupby("Month")["Amount"].sum()
monthly

In [ ]:
import matplotlib.pyplot as plt
monthly.plot(kind='bar',figsize=(10,5),color="steelblue")
plt.title("Monthly Spending")
plt.xlabel("Month")
plt.ylabel("Amount (INR)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
flow = df.pivot_table(index="Month", columns="Transaction Type", values="Amount", aggfunc="sum")
flow["NET"] = flow["CREDIT"] - flow["DEBIT"]
print(flow)

flow[["CREDIT", "DEBIT"]].plot(kind="line", marker="o", figsize=(10, 5))
plt.title("Money In vs Money Out")
plt.xlabel("Month")
plt.ylabel("Amount (INR)")
plt.tight_layout()
plt.show()

In [ ]:
df["Payee"] = (df["Transaction Details"]
    .str.replace(r"^(Paid to|Received from)\s*", "", regex=True)
    .str.strip())

rules = {
    "MSRTC": "Transport",
    "KIRANA": "Groceries",
    "MEESHO": "Shopping",
    "FLIPKART": "Shopping",
    "RESTAURANT": "Food",
    "HOTEL": "Food",
}

def categorize(payee):
    for keyword, category in rules.items():
        if keyword in payee.upper():
            return category
    return "Other"

df["Category"] = df["Payee"].apply(categorize)
print(df["Category"].value_counts())

In [ ]:
spending = df[df["Transaction Type"] == "DEBIT"]
other = spending[spending["Category"] == "Other"]
top = other.groupby("Payee")["Amount"].agg(["count", "sum"]).sort_values("sum", ascending=False)
top.head(20)

In [ ]:
import unicodedata

df["Payee"] = df["Payee"].apply(lambda s: unicodedata.normalize("NFKC", s))

rules.update({
    "SCHOOL": "Education",
    "COLLEGE B": "Education",
    "PERSON 1": "Family & Friends",
    "PERSON 2": "Family & Friends",
    "PERSON 3": "Family & Friends",
    "PERSON 4": "Family & Friends",
    "PERSON 5": "Family & Friends",
    "PERSON 6": "Family & Friends",
    "PERSON 7": "Family & Friends",
    "PERSON 8": "Family & Friends",
    "PERSON 9": "Family & Friends",
    "PERSON 10": "Family & Friends",
    "PERSON 11": "Family & Friends",
    "PERSON 12": "Family & Friends",
    "PERSON 13": "Family & Friends",
})

df["Category"] = df["Payee"].apply(categorize)
print(df["Category"].value_counts())

In [ ]:
spending = df[df["Transaction Type"] == "DEBIT"]

by_cat = spending.pivot_table(index="Month", columns="Category", values="Amount", aggfunc="sum", fill_value=0)
print(by_cat)

by_cat.plot(kind="bar", stacked=True, figsize=(12, 6))
plt.title("Monthly Spending by Category")
plt.xlabel("Month")
plt.ylabel("Amount (INR)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
june=spending[spending["Month"]=="2026-06"]
june.groupby("Category")["Amount"].sum().sort_values(ascending=False)

In [ ]:
edu=spending[spending["Category"]=="Education"].groupby("Month")["Amount"].sum()
flow["EDU"]=edu
flow["EDU"]=flow["EDU"].fillna(0)
flow["NET_without_edu"]=flow["NET"]+flow["EDU"]
print(flow[["NET","EDU","NET_without_edu"]])

